# Data Science Tool Mastery Project
**Data Science Consultant Internship — Task 4**

This notebook demonstrates proficiency across the core data science toolchain — Python, NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, and SQL — using the Breast Cancer Wisconsin (Diagnostic) dataset (carried forward, cleaned, from Task 3). A Power BI dashboard is specified at the end for business-facing reporting.

## 1. Master Python & Data Science Libraries
Core libraries imported and dataset loaded.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# Cleaned dataset from Task 3 (569 rows, no missing values/duplicates)
df = pd.read_csv('breast_cancer_clean.csv')
print(df.shape)
df.head()

## 2. Practice Data Analysis (Pandas & NumPy)
Transformation, aggregation, and NumPy-based computation on top of the already-cleaned dataset.

In [ ]:
feature_cols = ['radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean',
                'compactness_mean','concavity_mean','symmetry_mean']

# Aggregation: mean/std by diagnosis
agg = df.groupby('diagnosis')[feature_cols].agg(['mean','std']).round(2)
agg

In [ ]:
# NumPy: z-score normalization of radius_mean, and a vectorized flag for values > 1.5 std from the mean
radius = df['radius_mean'].to_numpy()
z_scores = (radius - np.mean(radius)) / np.std(radius)
df['radius_is_extreme'] = np.abs(z_scores) > 1.5
print("Extreme radius cases:", df['radius_is_extreme'].sum())

# Transformation: log-scale area (useful since area is right-skewed)
df['area_log'] = np.log1p(df['area_mean'])
df[['area_mean','area_log','radius_is_extreme']].head()

## 3. Build Data Visualizations
Meaningful charts built with Matplotlib and Seaborn.

In [ ]:
# Boxplot: area_mean by diagnosis
plt.figure(figsize=(6,5.5))
sns.boxplot(data=df, x='diagnosis', y='area_mean', palette={'benign':'#55A868','malignant':'#C44E52'})
plt.title('Area Mean by Diagnosis')
plt.show()

In [ ]:
# Pairwise relationship: radius vs texture, colored by diagnosis
plt.figure(figsize=(7,5.5))
for d,c in {'benign':'#55A868','malignant':'#C44E52'}.items():
    sub = df[df['diagnosis']==d]
    plt.scatter(sub['radius_mean'], sub['texture_mean'], label=d, alpha=0.6, color=c, edgecolor='k')
plt.xlabel('Radius Mean'); plt.ylabel('Texture Mean')
plt.title('Radius vs Texture by Diagnosis')
plt.legend()
plt.show()

## 4. Apply Machine Learning (Scikit-learn)
Preprocessing, model building, evaluation, and interpretation of a diagnosis classifier.

In [ ]:
# Define features/target and split into train/test sets
X = df[feature_cols]
y = (df['diagnosis'] == 'malignant').astype(int)  # 1 = malignant, 0 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing: standardize features (mean 0, std 1) — required for logistic regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

In [ ]:
# Model building: Logistic Regression classifier
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

# Evaluation
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['benign','malignant']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5.5,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['benign','malignant'], yticklabels=['benign','malignant'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

In [ ]:
# Interpretation: standardized coefficients show each feature's impact on predicting malignancy
coefs = pd.Series(model.coef_[0], index=feature_cols).sort_values()
plt.figure(figsize=(7,5))
colors = ['#C44E52' if v < 0 else '#4C72B0' for v in coefs.values]
plt.barh(coefs.index, coefs.values, color=colors)
plt.title('Logistic Regression Feature Coefficients (standardized)')
plt.xlabel('Coefficient (impact on malignant prediction)')
plt.show()
coefs.sort_values(key=abs, ascending=False).round(3)

**Interpretation:** radius_mean, area_mean, texture_mean, and perimeter_mean have the largest positive coefficients — larger, coarser-textured tumors are predicted more likely malignant. compactness_mean has a negative coefficient once the other size features are controlled for, meaning its signal is mostly already captured by radius/area/perimeter. The model reaches 93% accuracy and 93% recall on malignant cases, which is the more clinically important metric since missing a true malignant case (a false negative) is more costly than a false positive.

## 5. Work with SQL
The cleaned dataset is loaded into a SQLite database and queried with SQL — the same engine/approach used for structured data analysis and to feed a Power BI dashboard.

In [ ]:
conn = sqlite3.connect('tumors.db')
df.drop(columns=['radius_is_extreme','area_log']).to_sql('tumors', conn, if_exists='replace', index=False)

query1 = """
SELECT diagnosis,
       ROUND(AVG(radius_mean),2) AS avg_radius,
       ROUND(AVG(area_mean),2) AS avg_area,
       COUNT(*) AS n_cases
FROM tumors
GROUP BY diagnosis;
"""
pd.read_sql(query1, conn)

In [ ]:
query2 = """
SELECT radius_category,
       ROUND(100.0 * SUM(CASE WHEN diagnosis='malignant' THEN 1 ELSE 0 END) / COUNT(*), 1) AS malignant_pct,
       COUNT(*) AS n
FROM tumors
GROUP BY radius_category
ORDER BY malignant_pct DESC;
"""
pd.read_sql(query2, conn)

In [ ]:
query3 = """
SELECT radius_mean, area_mean, perimeter_mean, diagnosis
FROM tumors
WHERE diagnosis = 'malignant'
ORDER BY area_mean DESC
LIMIT 5;
"""
pd.read_sql(query3, conn)

conn.close()

## 6. Power BI Dashboard & Portfolio Project

**Power BI dashboard (build spec):** the cleaned dataset was exported as `breast_cancer_for_powerbi.csv` for direct import into Power BI Desktop. Recommended dashboard layout:
- **KPI cards (top row):** Total cases (569), Malignant % (37.3%), Benign % (62.7%), Model accuracy (93%).
- **Bar chart:** Malignancy rate by `radius_category` (Small → Very Large) — mirrors the SQL query above.
- **Scatter chart:** `radius_mean` vs `concavity_mean`, colored by `diagnosis`, with a slicer on `radius_category`.
- **Donut chart:** case split by `diagnosis`.
- **Table/matrix:** average of each `_mean` feature grouped by `diagnosis`, for drill-through.
- **Slicers:** `diagnosis`, `radius_category` to let a viewer filter every visual at once.

*(Power BI Desktop itself isn't available to run in this notebook environment — the CSV above is ready to import, and following this layout in Power BI reproduces the same views as the charts and SQL results already in this notebook.)*

**Portfolio project (Jupyter + Git/GitHub):** this notebook, the Word report, the exported CSV, and the SQLite database should be committed to a GitHub repository with a README describing the dataset, workflow, model results (93% accuracy / 93% recall), and the Power BI dashboard spec above — making this a complete, reviewable, end-to-end portfolio piece.